**Machine Learning Lab**

# 9. Image Classification Using Conventional ML

Neural networks aren't the only way to classify images — this notebook shows how far you can get with **conventional ML classifiers** (Logistic Regression, Random Forest, SVM) by first turning each image into a hand-crafted numeric feature vector (pixel values, edges, and shape moments), then feeding those features into a standard classifier. This approach predates deep learning and is still useful for small datasets or when you need an interpretable, lightweight model.

Import the core libraries for data handling and plotting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Loading the Dataset

For this example, we'll use the digits dataset built into scikit-learn — 1,797 images of handwritten digits (0–9), each an 8x8 grayscale image. It ships with scikit-learn, so no upload or download step is needed.

In [ ]:
# For this example, we'll use the digits dataset from sklearn
from sklearn.datasets import load_digits

digits = load_digits()
X = digits.images
y = digits.target

Check the shapes, and look at one example image with its true label.

In [ ]:
print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

#Displaying and image from the dataset
plt.gray()
print(f"Target label: {digits.target[800]}")
plt.matshow(digits.images[800])
plt.show()

## Feature Extraction

For each image we build a feature vector combining three things:
1. **Raw pixel values** — the flattened 8x8 grid of intensities.
2. **Edges** — detected with the Canny edge detector, which highlights the outline/strokes of the digit.
3. **Moments and Hu moments** — statistical descriptors of the image's shape. Regular moments capture things like the shape's centroid and spread; Hu moments are a set of 7 values derived from them that stay the same even if the shape is rotated, scaled, or translated, which makes them a useful compact shape signature.

This needs OpenCV (`cv2`); install it first if it isn't already available.

In [ ]:
# Feature extraction from images
import cv2

def extract_features(image):
  features = []

  # Original pixel values
  features.extend(image.flatten())

  # Canny edges. This dataset's pixel values only range 0-16, which is too dim for Canny's default
  # thresholds to detect anything, so we first rescale the image to the full 0-255 range.
  img_uint8 = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
  edges = cv2.Canny(img_uint8, 100, 200)
  features.extend(edges.flatten())

  # Moments
  moments = cv2.moments(image)
  features.extend(moments.values())

  # Hu moments: 7 rotation/scale/translation-invariant shape descriptors derived from the moments
  # above. (These were being computed but discarded in an earlier version of this notebook -
  # now they're actually added to the feature vector.)
  hu_moments = cv2.HuMoments(moments).flatten()
  features.extend(hu_moments)

  return features

Apply feature extraction to every image in the dataset.

In [ ]:
# Apply feature extraction to all images
X_features = np.array([extract_features(img) for img in X])

print(f"Shape of X_features: {X_features.shape}")

Take a look at the resulting feature matrix — one row per image, one column per feature.

In [ ]:
X_features

## Preparing the Data for Classification

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

Split into train/test sets, then standardize the features (subtract mean, divide by standard deviation) so that features on very different scales — e.g. raw pixel values vs. Hu moments — don't dominate the model just because their numbers happen to be larger.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_features, y, test_size=0.2, random_state=42)

# Standardizing the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Our feature vector is fairly high-dimensional (pixels + edges + moments). `PCA` compresses it down to 20 components that capture most of the variance, which speeds up training and can reduce overfitting.

In [ ]:
pcs = PCA(n_components=20)
X_train_pca = pcs.fit_transform(X_train)
X_test_pca = pcs.transform(X_test)

## Training and Evaluating Classifiers

Define a small helper that trains a classifier and prints its accuracy, confusion matrix, and classification report, so we can reuse it for each model below.

In [ ]:
def evaluate_classifier(clf, X_train, X_test, y_train, y_test):
  clf.fit(X_train, y_train)
  y_pred = clf.predict(X_test)
  print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
  print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
  print(f"Classification Report:\n{classification_report(y_test, y_pred)}")
  plt.show()

Import three classic classifiers to compare.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

Train and evaluate each one on the same PCA-reduced features.

In [ ]:
# Initialize and train classifiers
logistic_regression = LogisticRegression(max_iter=10000)  # Increased max_iter
evaluate_classifier(logistic_regression, X_train_pca, X_test_pca, y_train, y_test)

random_forest = RandomForestClassifier()
evaluate_classifier(random_forest, X_train_pca, X_test_pca, y_train, y_test)

svm = SVC()
evaluate_classifier(svm, X_train_pca, X_test_pca, y_train, y_test)

## Practice Problems

1. Try changing `PCA(n_components=20)` to a few other values (e.g. 5, 50). How does accuracy for each classifier change as you keep more or fewer components?
2. Skip PCA entirely and train the classifiers directly on the standardized `X_train`/`X_test` (before the PCA step). Is accuracy better, worse, or about the same? What about training time?
3. Modify `extract_features` to drop the Canny edge features (just pixels + moments + Hu moments). Does removing them hurt accuracy much?
4. Pick one misclassified test image (compare `y_test` to each model's predictions) and display it with `plt.matshow`. Does it look genuinely ambiguous to your own eye?
5. Try `KNeighborsClassifier` as a fourth model using the same `evaluate_classifier` helper. How does it compare to the three models already here?